In [ ]:
# !pip install langchain-openai
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.tools import tool
from langchain_core.messages import ToolMessage

#Challenge 1: Dynamic Persona Chat Prompt Template

**Task**: Construct a `Chat Prompt Template` named `persona_prompt` that includes:
- A `system` message establishing a persona `({persona})` and difficulty target `({target_level})`.
- A `human` message asking to explain a technical topic `({topic})`.

Then build an LCEL chain using `persona_prompt | 11m | StroutputParser()` and invoke it with:
- `persona`:
`"Quantum Physicist"`
- `target_level`: `"high school student"`
- `topic`: `"Quantum Superposition"`

In [ ]:
OPENROUTER_API_KEY = "YOUR_API_KEY"
MODEL_NAME = "cohere/north-mini-code:free"

llm = ChatOpenAI(
    model=MODEL_NAME,
    openai_api_key=OPENROUTER_API_KEY,
    openai_api_base="https://openrouter.ai/api/v1",
    temperature=0.2
)

persona_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a {persona}. Explain concepts for a {target_level}."),
    ("human", "Explain {topic}.")
])

# LCEL Chain
parser = StrOutputParser()
simple_chain = persona_prompt | llm | parser

print("--- LCEL Chain Result ---")
result1 = simple_chain.invoke({
        "persona": "Quantum Physicist",
        "target_level": "high school student",
        "topic": "Quantum Superposition"
    })
print(result1)

--- LCEL Chain Result ---
**Quantum Superposition – What It Is and Why It’s Weird**

---

### 1. The Classic “Both‑At‑Once” Picture

In everyday life we never see a thing doing two different things at the same time.  
A ball is either **up** or **down** on a table, a light switch is either **on** or **off**, and a person is either **standing** or **sitting**.

Quantum physics, however, allows a tiny particle (like an electron, photon, or even a larger system such as a molecule) to exist in a *combination* of several possibilities at once.  
We call this combination a **superposition**.

> **Superposition**: A quantum state that is a weighted blend of two or more simpler states.  
> Think of it as a recipe that mixes together several ingredients, where each ingredient represents a possible outcome.

---

### 2. A Simple Analogy: Musical Chords

Imagine a guitar string that can vibrate in two distinct ways:

| State | What it sounds like |
|-------|---------------------|
| **A** | A pure

#Challenge 2: Multi-Stage Sequential LCEL Chain

**Task**: Construct a 2-stage sequential chain:
Stage 1 (`summarize_chain`): Takes `{article_text}` and summarizes it into 2 bullet points.
Stage 2 (`translate_chain`): Takes the summary from Stage 1 and translates it into French.
Combine them using LCEL pipe operator: `{"summary": summarize_chain} | translate_prompt | 1lm | StroutputParser()`.

In [ ]:
summarize_chain = PromptTemplate.from_template(
    """
    Summarize the following article into exactly 2 bullet points
    Article: {article_text}
    """
)
summarize_chain = summarize_chain | llm | parser

translate_chain = PromptTemplate.from_template(
    """
    Translate the following summary into French
    Summary:
    {summary}
    """
)
sequential_chain = {"summary": summarize_chain} | translate_chain | llm | parser

print("\n--- Sequential Chain Output ---")
result2 = sequential_chain.invoke({"article_text": "JWT Authentication Middleware in FastAPI"})
print(result2)


--- Sequential Chain Output ---
Résumé :
- Le middleware d'authentification JWT de FastAPI propose une dépendance réutilisable qui valide les jetons JWT, extrait les informations utilisateur et injecte les données de l'utilisateur authentifié dans les gestionnaires de routes.
- L'article explique comment créer une classe JWTBearer personnalisée (ou utiliser les utilitaires JWT intégrés), configurer la vérification des jetons, gérer les erreurs et intégrer le middleware dans les applications FastAPI pour une authentification sécurisée basée sur les jetons.


#Challenge 3: Financial Tools & Tool Binding

**Task**: Define two custom Python tools using @tool:
1. `currency_converter (amount: float, from_curr: str, to_curr: str) -> float:` converts amount between USD, EUR, GBP.
Use exchange rates: USD_EUR-0.92, USD_GBP=0.79, EUR_USD=1.09, GBP_USD=1.26.
2. `calculate_compound_interest (principal: float, rate: float, years: int) -> float`: calculates compound interest `principal * ((1 + rate) ** years)`.
Bind both tools to 11m as 11m_with_fin_tools and invoke a prompt requiring tool selection.

In [ ]:
@tool
def currency_converter (amount: float, from_curr: str, to_curr: str) -> float:
  """
    Converts an amount between USD, EUR, and GBP using fixed exchange rates.
    Supported conversions:
    - USD -> EUR
    - USD -> GBP
    - EUR -> USD
    - GBP -> USD
    """
  from_cur = from_curr.upper()
  to_cur = to_curr.upper()

  if(from_curr == "USD"):
    if(to_curr == "EUR"):
      new_amount = amount * 0.92
    elif(to_curr == "GBP"):
      new_amount = amount * 0.79
  elif(from_curr == "EUR"):
    if(to_curr == "USD"):
      new_amount = amount * 1.09
  elif(from_curr == "GBP"):
    if(to_curr == "USD"):
      new_amount = amount * 1.26
  else:
    new_amount = "Conversion not supported"

  return new_amount

@tool
def calculate_compound_interest(principal: float, rate: float, years: int) -> float:
  """
    Calculates compound interest using the formula:
    Final Amount = Principal × (1 + Rate)^Years.

    The interest rate should be provided as a decimal
    (e.g., 0.05 for 5%).
    """
  return principal * (1 + rate) ** years

tools = [currency_converter, calculate_compound_interest]
print("Registered Tools:", [t.name for t in tools])

llm_with_tools = llm.bind_tools(tools)

Registered Tools: ['currency_converter', 'calculate_compound_interest']


#Challenge 4: End-to-End Tool Execution Agent Function

**Task**: Write a Python function `run_financial_agent(user_query: str) -> str` that:
1. Calls `11m_with_fin_tools.invoke(messages)`.
2. Checks if `response.tool_calls` exists.
If tool calls exist, iterates through each tool call, executes the tool function from `fin_tools`, appends a `ToolMessage` to the message history, and calls
3. `1lm_with_fin_tools.invoke(messages)` to return the final natural language answer.
4. If no tool calls exist, returns `response.content`.

In [ ]:
tool_map = {t.name: t for t in tools}

def run_financial_agent(user_query: str) -> str:
    messages = [HumanMessage(content=user_query)]

    response = llm_with_tools.invoke(messages)

    if response.tool_calls:
        messages.append(response)
        for tool_call in response.tool_calls:
            tool_name = tool_call["name"]
            tool_args = tool_call["args"]

            tool_function = tool_map[tool_name]
            tool_result = tool_function.invoke(tool_args)

            messages.append(
                ToolMessage(
                    content=str(tool_result),
                    tool_call_id=tool_call["id"]
                )
            )

        final_response = llm_with_tools.invoke(messages)

        return final_response.content

    return response.content

In [ ]:
print(run_financial_agent("How much is 250 USD in EUR?"))

250 USD is equal to 230.00 EUR at the current fixed exchange rate.


In [ ]:
print(run_financial_agent("Calculate compound interest for $5000 at 8% for 5 years."))

The compound interest calculation shows that **$5,000 invested at 8% annual interest for 5 years will grow to $7,346.64**.

This means:
- **Initial principal:** $5,000
- **Final amount:** $7,346.64
- **Total interest earned:** $2,346.64

The compound interest formula used was: Final Amount = Principal × (1 + Rate)^Years
= $5,000 × (1 + 0.08)^5
= $5,000 × (1.08)^5
= $5,000 × 1.46933
= $7,346.64

Your investment would grow by approximately 46.93% over the 5-year period due to the power of compound interest.


In [ ]:
print(run_financial_agent("Hello! How are you?"))

Hello! I'm doing well, thank you for asking! I'm here and ready to help you with any calculations or currency conversions you might need. How can I assist you today?
